<a href="https://colab.research.google.com/github/ndobrylko/ds-learning/blob/main/Lekcja_40_Zad_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
#instalacja bibliotek
!pip install fastapi uvicorn pyngrok nest-asyncio -q

In [2]:
#trening i zapis 3 moddeli
import os
import joblib

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()

X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "v1_logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "v2_random_forest": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))
    ]),
    "v3_gradient_boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
    ])
}

os.makedirs("models", exist_ok=True)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    path = f"models/{name}.pkl"
    joblib.dump(model, path)

    print(f"{name}: accuracy={acc:.4f}, zapisano do {path}")

v1_logistic_regression: accuracy=0.9333, zapisano do models/v1_logistic_regression.pkl
v2_random_forest: accuracy=0.9333, zapisano do models/v2_random_forest.pkl
v3_gradient_boosting: accuracy=0.9667, zapisano do models/v3_gradient_boosting.pkl


In [4]:
#stworzenie pliku API
%%writefile main.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import numpy as np

app = FastAPI(
    title="Multi-model Iris Prediction API",
    description="API obsługujące 3 wersje modelu ML",
    version="1.0.0"
)

MODEL_PATHS = {
    "v1": "models/v1_logistic_regression.pkl",
    "v2": "models/v2_random_forest.pkl",
    "v3": "models/v3_gradient_boosting.pkl"
}

models = {}

for version, path in MODEL_PATHS.items():
    try:
        models[version] = joblib.load(path)
        print(f"Wczytano model {version}: {path}")
    except FileNotFoundError:
        models[version] = None
        print(f"Nie znaleziono modelu {version}: {path}")

CLASS_NAMES = {
    0: "setosa",
    1: "versicolor",
    2: "virginica"
}

class IrisInput(BaseModel):
    sepal_length: float = Field(..., gt=0)
    sepal_width: float = Field(..., gt=0)
    petal_length: float = Field(..., gt=0)
    petal_width: float = Field(..., gt=0)

class PredictionResponse(BaseModel):
    model_version: str
    model_name: str
    prediction: int
    prediction_label: str
    probability: float

def make_prediction(data: IrisInput, version: str, model_name: str):
    model = models.get(version)

    if model is None:
        raise HTTPException(
            status_code=503,
            detail=f"Model {version} nie jest dostępny"
        )

    features = np.array([[
        data.sepal_length,
        data.sepal_width,
        data.petal_length,
        data.petal_width
    ]])

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][prediction])

    return PredictionResponse(
        model_version=version,
        model_name=model_name,
        prediction=prediction,
        prediction_label=CLASS_NAMES[prediction],
        probability=round(probability, 4)
    )

@app.get("/")
def root():
    return {
        "message": "Multi-model Iris Prediction API",
        "available_versions": {
            "v1": "LogisticRegression",
            "v2": "RandomForest",
            "v3": "GradientBoosting"
        },
        "docs": "/docs"
    }

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "models_loaded": {
            version: model is not None
            for version, model in models.items()
        }
    }

@app.post("/predict/v1", response_model=PredictionResponse)
def predict_v1(data: IrisInput):
    return make_prediction(data, "v1", "LogisticRegression")

@app.post("/predict/v2", response_model=PredictionResponse)
def predict_v2(data: IrisInput):
    return make_prediction(data, "v2", "RandomForest")

@app.post("/predict/v3", response_model=PredictionResponse)
def predict_v3(data: IrisInput):
    return make_prediction(data, "v3", "GradientBoosting")

@app.post("/predict/compare")
def compare_models(data: IrisInput):
    return {
        "input": data.model_dump(),
        "predictions": [
            predict_v1(data),
            predict_v2(data),
            predict_v3(data)
        ]
    }

Overwriting main.py


In [10]:
#uruchomienie API w Collabie
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run_api():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

INFO:     Started server process [13432]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [11]:
#test lokalny w Colabie
import requests

url = "http://127.0.0.1:8000/health"

response = requests.get(url)
response.json()

INFO:     127.0.0.1:40044 - "GET /health HTTP/1.1" 200 OK


{'status': 'healthy', 'models_loaded': {'v1': True, 'v2': True, 'v3': True}}

In [12]:
#test predykcji jednego modelu
sample = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = requests.post(
    "http://127.0.0.1:8000/predict/v1",
    json=sample
)

response.json()

INFO:     127.0.0.1:50270 - "POST /predict/v1 HTTP/1.1" 200 OK


{'model_version': 'v1',
 'model_name': 'LogisticRegression',
 'prediction': 0,
 'prediction_label': 'setosa',
 'probability': 0.9808}

In [13]:
#porównanie 3 modeli
response = requests.post(
    "http://127.0.0.1:8000/predict/compare",
    json=sample
)

response.json()

INFO:     127.0.0.1:55970 - "POST /predict/compare HTTP/1.1" 200 OK


{'input': {'sepal_length': 5.1,
  'sepal_width': 3.5,
  'petal_length': 1.4,
  'petal_width': 0.2},
 'predictions': [{'model_version': 'v1',
   'model_name': 'LogisticRegression',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 0.9808},
  {'model_version': 'v2',
   'model_name': 'RandomForest',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 1.0},
  {'model_version': 'v3',
   'model_name': 'GradientBoosting',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 1.0}]}

In [17]:
#test API lokalnie w Colabie
import requests

sample = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

response = requests.post(
    "http://127.0.0.1:8000/predict/compare",
    json=sample
)

response.json()

INFO:     127.0.0.1:47766 - "POST /predict/compare HTTP/1.1" 200 OK


{'input': {'sepal_length': 5.1,
  'sepal_width': 3.5,
  'petal_length': 1.4,
  'petal_width': 0.2},
 'predictions': [{'model_version': 'v1',
   'model_name': 'LogisticRegression',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 0.9808},
  {'model_version': 'v2',
   'model_name': 'RandomForest',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 1.0},
  {'model_version': 'v3',
   'model_name': 'GradientBoosting',
   'prediction': 0,
   'prediction_label': 'setosa',
   'probability': 1.0}]}

Aplikacja FastAPI obsługuje trzy modele ML:

* LogisticRegression (/predict/v1)
* RandomForest (/predict/v2)
* GradientBoosting (/predict/v3)

Modele zostały wytrenowane na zbiorze Iris i zapisane jako pliki .pkl.

API wykorzystuje Pydantic do walidacji danych wejściowych.

Dodatkowo zaimplementowano endpoint /predict/compare, który zwraca predykcje wszystkich modeli jednocześnie.
